# SVM Accelerator — PYNQ Z2 Deployment

This notebook deploys the **Support Vector Machine (RBF kernel, One-vs-One)** HLS accelerator
on the PYNQ Z2 FPGA and benchmarks it across all 4 datasets: Iris, Wine, Cancer, MNIST.

**Architecture:** Support vectors stored transposed in BRAM for pipelined kernel evaluation (II=1).
RBF kernel computed via `exp_approx()` using a 256-entry LUT. OvO voting for multi-class.

**Scaling:** SVM uses MinMax-scaled features (not global-scaled).

**Interface:** AXI-Stream for data transfer (mode 0 = load model, mode 1 = inference),
AXI-Lite for control registers.

In [ ]:
import time, struct
import numpy as np
from pynq import Overlay, allocate

AP_CTRL  = 0x00
AP_START = 0x01
AP_DONE  = 0x02

REG_MODE         = 0x10
REG_NUM_FEATURES = 0x18
REG_NUM_CLASSES  = 0x20
REG_N_SV         = 0x28
REG_NUM_TEST     = 0x30
REG_LATENCY_OUT  = 0x38

FRAC_BITS = 8
SCALE = 1 << FRAC_BITS

def to_fixed(val):
    v = int(round(val * SCALE))
    return np.int32(max(-32768, min(32767, v)))

def to_u32(v):
    return np.uint32(struct.unpack('<I', struct.pack('<i', int(v)))[0])

def write_reg(ip, off, val):
    ip.write(off, int(val))

def read_reg(ip, off):
    return ip.read(off)

print("Utilities loaded.")

In [ ]:
def load_svm_model(ip, dma, params):
    """Mode 0: Stream SVM parameters.
    Order: gamma, n_support[], intercept[], SVs (row-major), dual_coef (row-major)
    """
    gamma = params["gamma"]
    n_support = params["n_support"].astype(np.int32)
    intercept = params["intercept"].astype(np.int32)
    svs = params["support_vectors"].astype(np.int32)
    dc = params["dual_coef"].astype(np.int32)
    n_classes = int(params["n_classes"])
    n_sv = int(params["n_sv"])
    n_feat = int(params["n_features"])

    write_reg(ip, REG_MODE, 0)
    write_reg(ip, REG_NUM_FEATURES, n_feat)
    write_reg(ip, REG_NUM_CLASSES, n_classes)
    write_reg(ip, REG_N_SV, n_sv)
    write_reg(ip, REG_NUM_TEST, 0)

    words = []
    # 1) gamma (fix16)
    words.append(to_u32(to_fixed(float(gamma))))
    # 2) n_support per class
    for c in range(n_classes):
        words.append(to_u32(n_support[c]))
    # 3) intercept
    for v in intercept.flat:
        words.append(to_u32(v))
    # 4) support vectors (row-major)
    for v in svs.flat:
        words.append(to_u32(v))
    # 5) dual_coef (row-major)
    for v in dc.flat:
        words.append(to_u32(v))

    buf_in = allocate(shape=(len(words),), dtype=np.uint32)
    buf_in[:] = np.array(words, dtype=np.uint32)
    buf_in.flush()

    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(buf_in)
    dma.sendchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass

    buf_in.freebuffer()
    print(f"  Loaded SVM: {n_sv} SVs, {n_feat} features, {n_classes} classes")
    return n_feat, n_classes


def run_svm_inference(ip, dma, X_test, y_true, n_feat, n_classes):
    """Mode 1: Stream test features (MinMax-scaled), receive predicted classes."""
    n_test = len(y_true)
    write_reg(ip, REG_MODE, 1)
    write_reg(ip, REG_NUM_TEST, n_test)

    words = []
    for i in range(n_test):
        for j in range(n_feat):
            words.append(to_u32(X_test[i, j]))
    buf_in = allocate(shape=(len(words),), dtype=np.uint32)
    buf_out = allocate(shape=(n_test,), dtype=np.uint32)
    buf_in[:] = np.array(words, dtype=np.uint32)
    buf_in.flush()

    t0 = time.perf_counter()
    write_reg(ip, AP_CTRL, AP_START)
    dma.sendchannel.transfer(buf_in)
    dma.recvchannel.transfer(buf_out)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    while not (read_reg(ip, AP_CTRL) & AP_DONE): pass
    t1 = time.perf_counter()

    buf_out.invalidate()
    preds = np.frombuffer(buf_out, dtype=np.int32)[:n_test].copy()
    hw_cycles = read_reg(ip, REG_LATENCY_OUT)
    wall_us = (t1 - t0) * 1e6
    acc = np.sum(preds == y_true) / n_test * 100.0

    buf_in.freebuffer(); buf_out.freebuffer()
    return {"accuracy": acc, "hw_cycles": hw_cycles, "wall_us": wall_us, "preds": preds}

print("SVM functions loaded.")

## Load Overlay

In [ ]:
ol = Overlay("overlays/svm_accel.bit")
ip = ol.svm_accel_0
dma = ol.axi_dma_0
print("SVM overlay loaded.")
print("IP registers:", ip.register_map)

## Run on All Datasets

In [ ]:
DATASETS = ["iris", "wine", "cancer", "mnist"]
FPGA_CLK_MHZ = 100.0
results = []

for ds in DATASETS:
    print(f"\n{'='*50}")
    print(f"  SVM on {ds.upper()}")
    print(f"{'='*50}")

    params = np.load(f"models/{ds}/svm_params.npz", allow_pickle=True)
    test   = np.load(f"test_vectors/{ds}/test_data.npz", allow_pickle=True)

    ol = Overlay("overlays/svm_accel.bit")
    ip = ol.svm_accel_0
    dma = ol.axi_dma_0

    n_feat, n_cls = load_svm_model(ip, dma, params)

    n_test = int(test["n_test"])
    X_test = test["X_test_sc_q"][:n_test]  # SVM uses MinMax-scaled data
    y_true = test["y_test"][:n_test]

    r = run_svm_inference(ip, dma, X_test, y_true, n_feat, n_cls)
    r["dataset"] = ds
    r["fpga_latency_us"] = r["hw_cycles"] / FPGA_CLK_MHZ
    results.append(r)

    print(f"  Accuracy:       {r['accuracy']:.2f}%")
    print(f"  HW Cycles:      {r['hw_cycles']}")
    print(f"  FPGA Latency:   {r['fpga_latency_us']:.2f} µs")
    print(f"  Wall Time:      {r['wall_us']:.1f} µs")

## Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame([{"Dataset": r["dataset"], "Accuracy (%)": r["accuracy"],
                     "HW Cycles": r["hw_cycles"],
                     "FPGA Latency (µs)": r["fpga_latency_us"],
                     "Wall Time (µs)": r["wall_us"]} for r in results])
print("\nSVM Accelerator Results:")
print(df.to_string(index=False))
df.to_csv("svm_results.csv", index=False)
print("\nSaved to svm_results.csv")

## FPGA vs ESP32 Comparison

In [ ]:
esp32_svm = {"iris": None, "wine": None, "cancer": None, "mnist": None}

print(f"{'Dataset':<10} {'FPGA(µs)':>10} {'ESP32(µs)':>10} {'Speedup':>10}")
print("-" * 42)
for r in results:
    ds = r["dataset"]
    fpga_us = r["fpga_latency_us"]
    esp_us = esp32_svm.get(ds)
    speedup = f"{esp_us/fpga_us:.1f}x" if esp_us else "X"
    print(f"{ds:<10} {fpga_us:>10.2f} {str(esp_us or 'X'):>10} {speedup:>10}")